# Check INT8 Model Size

Этот ноутбук оценивает размер модели после int8-квантизации для текущей архитектуры.

- Базовые значения взяты из `run_hid_dim_512_num_layers_9.sh`
- Можно вручную менять поля в объекте `Hyperparameters` в следующей ячейке
- После перебора ячейка пишет `launch_grid_generated/run_golf_*.sh` для `torchrun train_gpt.py`
- Следующая ячейка собирает `launch_grid_generated/run_all_golf_grid.sh` — по очереди все эти скрипты
- В **Comet** метрики `model_size_bytes` / `model_size_mb_ceil` — размер **сабмита** `final_model.int8.ptz` (int8 + **zlib**), как в конце `train_gpt.py`. Отдельно логируются `int8_payload_*` и `model_fp32_pt_*`.
- Перебор гиперпараметров ниже по-прежнему фильтрует по **`int8_payload_bytes`** (логический вес без zlib); zlib обычно чуть меньше raw `torch.save`.

In [1]:
import io
import math
import zlib

import torch

from train_gpt_lib.config import Hyperparameters
from train_gpt_lib.model import GPT
from train_gpt_lib.serialization import estimate_int8_payload_bytes_gpt, quantize_state_dict_int8


In [2]:
args = Hyperparameters()

args.vocab_size = 1024
args.num_layers = 12
args.model_dim = 352
args.num_heads = 8
args.num_kv_heads = 4
args.mlp_mult = 3
args.tie_embeddings = True
args.tied_embed_init_std = 0.005
args.logit_softcap = 30.0
args.rope_base = 10000.0
args.qk_gain_init = 1.5
args.use_mhc = False
args.mhc_type = "none"
args.mhc_num_streams = 1

print("Using config:")
for k in [
    "vocab_size", "num_layers", "model_dim", "num_heads", "num_kv_heads",
    "mlp_mult", "tie_embeddings", "use_mhc", "mhc_type", "mhc_num_streams"
]:
    print(f"  {k} = {getattr(args, k)}")

Using config:
  vocab_size = 1024
  num_layers = 12
  model_dim = 352
  num_heads = 8
  num_kv_heads = 4
  mlp_mult = 3
  tie_embeddings = True
  use_mhc = False
  mhc_type = none
  mhc_num_streams = 1


In [3]:
# Создаем модель, квантуем и считаем размер
model = GPT(
    vocab_size=args.vocab_size,
    num_layers=args.num_layers,
    model_dim=args.model_dim,
    num_heads=args.num_heads,
    num_kv_heads=args.num_kv_heads,
    mlp_mult=args.mlp_mult,
    tie_embeddings=args.tie_embeddings,
    tied_embed_init_std=args.tied_embed_init_std,
    logit_softcap=args.logit_softcap,
    rope_base=args.rope_base,
    qk_gain_init=args.qk_gain_init,
    hyper_conn_type=(args.mhc_type if args.use_mhc else "none"),
    hyper_conn_n=args.mhc_num_streams,
)

state_dict = model.state_dict()
quant_obj, quant_stats = quantize_state_dict_int8(state_dict)

# Полезный payload из quant_stats (int8 тензоры + scale + passthrough)
int8_payload_bytes = int(quant_stats["int8_payload_bytes"])

# Размер int8-объекта в torch-serialization (без zlib), как перед compress в train
buf = io.BytesIO()
torch.save(quant_obj, buf)
int8_serialized_raw = buf.getvalue()
int8_serialized_bytes = len(int8_serialized_raw)

# Как ``final_model.int8.ptz`` и Comet ``model_size_bytes`` / ``model_size_mb_ceil``
int8_zlib_bytes = len(zlib.compress(int8_serialized_raw, level=9))

int8_serialized_mb = int8_serialized_bytes / (1024 * 1024)
int8_payload_mb = int8_payload_bytes / (1024 * 1024)
int8_zlib_mb = int8_zlib_bytes / (1024 * 1024)
int8_serialized_mb_ceil = math.ceil(int8_serialized_mb)
int8_payload_mb_ceil = math.ceil(int8_payload_mb)
int8_zlib_mb_ceil = math.ceil(int8_zlib_mb)

print(f"MODEL_PARAMS: {sum(p.numel() for p in model.parameters())}")
print(f"INT8_PAYLOAD_BYTES: {int8_payload_bytes}")
print(f"INT8_PAYLOAD_MB: {int8_payload_mb:.4f}")
print(f"INT8_PAYLOAD_MB_CEIL: {int8_payload_mb_ceil}")
print(f"INT8_SERIALIZED_BYTES: {int8_serialized_bytes}")
print(f"INT8_SERIALIZED_MB: {int8_serialized_mb:.4f}")
print(f"INT8_SERIALIZED_MB_CEIL: {int8_serialized_mb_ceil}")
print(f"INT8_ZLIB_BYTES (Comet model_size_bytes): {int8_zlib_bytes}")
print(f"INT8_ZLIB_MB: {int8_zlib_mb:.4f}")
print(f"INT8_ZLIB_MB_CEIL (Comet model_size_mb_ceil): {int8_zlib_mb_ceil}")

print("payload<=15MiB:", int8_payload_mb <= 15)
print("zlib<=15MiB:", int8_zlib_mb <= 15)

MODEL_PARAMS: 13761184
INT8_PAYLOAD_BYTES: 15358080
INT8_PAYLOAD_MB: 14.6466
INT8_PAYLOAD_MB_CEIL: 15
INT8_SERIALIZED_BYTES: 15410793
INT8_SERIALIZED_MB: 14.6969
INT8_SERIALIZED_MB_CEIL: 15
True


In [4]:
# Перебор гиперпараметров: int8 payload (как INT8_PAYLOAD_BYTES выше) <= 15 MiB
# num_heads / num_kv_heads не перебираем — как в ячейке с Hyperparameters (поменяйте в base при необходимости)
# model_dim в сетке только если делится на num_heads и head_dim чётный (RoPE)
# Перед GPT: точная оценка int8 payload (как в quantize_state_dict_int8) без сборки модели; при use_mhc=True — пропускаем фильтр

MAX_INT8_BYTES = 15 * 1024 * 1024

# Сетка без «мелочи»: мало слоёв и маленький hidden не перебираем (границы правьте под себя)
NUM_LAYERS_RANGE = range(8, 33, 4)  # 8..32 слоя (было с 4)
MODEL_DIM_RANGE = range(256, 641, 64)  # 256..640 шаг 16
MLP_MULT_LIST = (3, 4, 5)

from tqdm.auto import tqdm

base = Hyperparameters()
base.vocab_size = 1024
base.tie_embeddings = True
base.tied_embed_init_std = 0.005
base.logit_softcap = 30.0
base.rope_base = 10000.0
base.qk_gain_init = 1.5
base.num_heads = 8
base.num_kv_heads = 4
base.use_mhc = False
base.mhc_type = "none"
base.mhc_num_streams = 1


def _model_dim_valid(md: int) -> bool:
    nh, nkv = base.num_heads, base.num_kv_heads
    if nh % nkv:
        return False
    if md % nh:
        return False
    if (md // nh) % 2:
        return False
    return True


n_md = sum(1 for md in MODEL_DIM_RANGE if _model_dim_valid(md))
n_cfg = n_md * len(NUM_LAYERS_RANGE) * len(MLP_MULT_LIST)
print(
    f"Оценка числа конфигураций (до фильтра по размеру): {n_cfg:,} "
    f"(num_heads={base.num_heads}, num_kv_heads={base.num_kv_heads}, подходящих model_dim: {n_md})"
)

results = []
errors = 0
skipped_precheck = 0  # оценка int8 payload > лимита, GPT не создавали

outer = []
for nl in NUM_LAYERS_RANGE:
    for md in MODEL_DIM_RANGE:
        if not _model_dim_valid(md):
            continue
        for mm in MLP_MULT_LIST:
            outer.append((nl, md, mm))

for nl, md, mm in tqdm(outer, desc="grid"):
    pre_b = estimate_int8_payload_bytes_gpt(
        base.vocab_size,
        nl,
        md,
        base.num_heads,
        base.num_kv_heads,
        mm,
        base.tie_embeddings,
        use_mhc=base.use_mhc,
    )
    if pre_b is not None and pre_b > MAX_INT8_BYTES:
        skipped_precheck += 1
        continue

    model = None
    try:
        model = GPT(
            vocab_size=base.vocab_size,
            num_layers=nl,
            model_dim=md,
            num_heads=base.num_heads,
            num_kv_heads=base.num_kv_heads,
            mlp_mult=mm,
            tie_embeddings=base.tie_embeddings,
            tied_embed_init_std=base.tied_embed_init_std,
            logit_softcap=base.logit_softcap,
            rope_base=base.rope_base,
            qk_gain_init=base.qk_gain_init,
            hyper_conn_type=(base.mhc_type if base.use_mhc else "none"),
            hyper_conn_n=base.mhc_num_streams,
        )
        nparams = sum(p.numel() for p in model.parameters())
        _, quant_stats = quantize_state_dict_int8(model.state_dict())
        b = int(quant_stats["int8_payload_bytes"])
    except Exception:
        errors += 1
    else:
        if b <= MAX_INT8_BYTES:
            results.append(
                {
                    "num_layers": nl,
                    "model_dim": md,
                    "mlp_mult": mm,
                    "num_heads": base.num_heads,
                    "num_kv_heads": base.num_kv_heads,
                    "nparams": nparams,
                    "int8_payload_bytes": b,
                    "int8_payload_mb": b / (1024 * 1024),
                }
            )
    finally:
        if model is not None:
            del model

results.sort(key=lambda r: (-r["nparams"], r["int8_payload_bytes"], r["model_dim"], r["num_layers"]))

print(
    f"Подходит под <= 15 MiB int8 payload: {len(results)} комбинаций "
    f"(отсечено до GPT: {skipped_precheck}, ошибок при сборке: {errors})"
)
print()
for r in results:
    print(
        f"L={r['num_layers']:2d} d={r['model_dim']:3d} mlp={r['mlp_mult']} "
        f"nh={r['num_heads']:2d} nkv={r['num_kv_heads']:2d} | "
        f"params={r['nparams']:,} int8_B={r['int8_payload_bytes']:,} ({r['int8_payload_mb']:.4f} MiB)"
    )


/home/jovyan/vasiliev/notebooks/parameter-golf/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Оценка числа конфигураций (до фильтра по размеру): 147 (num_heads=8, num_kv_heads=4, подходящих model_dim: 7)


grid: 100%|██████████| 147/147 [00:01<00:00, 111.26it/s]

Подходит под <= 15 MiB int8 payload: 16 комбинаций (отсечено до GPT: 131, ошибок при сборке: 0)

L= 8 d=448 mlp=3 nh= 8 nkv= 4 | params=14,925,632 int8_B=15,026,432 (14.3303 MiB)
L=12 d=320 mlp=4 nh= 8 nkv= 4 | params=13,861,856 int8_B=15,198,592 (14.4945 MiB)
L= 8 d=384 mlp=4 nh= 8 nkv= 4 | params=13,383,232 int8_B=13,476,096 (12.8518 MiB)
L=16 d=256 mlp=4 nh= 8 nkv= 4 | params=11,815,040 int8_B=15,059,456 (14.3618 MiB)
L=12 d=320 mlp=3 nh= 8 nkv= 4 | params=11,404,256 int8_B=12,733,312 (12.1434 MiB)
L= 8 d=384 mlp=3 nh= 8 nkv= 4 | params=11,023,936 int8_B=11,110,656 (10.5959 MiB)
L= 8 d=320 mlp=5 nh= 8 nkv= 4 | params=10,988,864 int8_B=11,885,824 (11.3352 MiB)
L=12 d=256 mlp=5 nh= 8 nkv= 4 | params=10,499,680 int8_B=12,939,648 (12.3402 MiB)
L=16 d=256 mlp=3 nh= 8 nkv= 4 | params=9,717,888 int8_B=12,954,112 (12.3540 MiB)
L= 8 d=320 mlp=4 nh= 8 nkv= 4 | params=9,350,464 int8_B=10,242,304 (9.7678 MiB)
L=12 d=256 mlp=4 nh= 8 nkv= 4 | params=8,926,816 int8_B=11,360,640 (10.8344 MiB)
L= 8 

In [5]:
# Генерация run-скриптов по списку `results` из предыдущей ячейки (сначала выполните перебор)
from __future__ import annotations

import os
import stat
from pathlib import Path

# Корень репозитория (ноутбук обычно запускают из каталога parameter-golf)
REPO_ROOT = Path.cwd().resolve()
LAUNCH_DIR = REPO_ROOT / "launch_grid_generated"
LAUNCH_DIR.mkdir(parents=True, exist_ok=True)

# Как в run_4_h100.sh / run_hid_dim_*.sh — поправьте под свой кластер
NPROC_PER_NODE = 4
DATA_PATH = "./data/datasets/fineweb10B_sp1024/"
TOKENIZER_PATH = "./data/tokenizers/fineweb_1024_bpe.model"
VOCAB_SIZE = 1024
COMET_ENABLE = "1"
# При генерации: подставится в .sh; иначе каждый run_golf_*.sh подхватит launch_grid_generated/.comet_api_key
COMET_API_KEY = os.environ.get("COMET_API_KEY", "").strip()
if not COMET_API_KEY:
    _kf = LAUNCH_DIR / ".comet_api_key"
    if _kf.is_file():
        COMET_API_KEY = _kf.read_text(encoding="utf-8").strip()
USE_COMPILE = "1"

# В каждый .sh: если COMET_API_KEY не в окружении — читаем launch_grid_generated/.comet_api_key
_NL = chr(10)

if "results" not in globals() or not globals()["results"]:
    raise RuntimeError("Сначала выполните ячейку с перебором: должен быть непустой список results")


def sh_quote(s: str) -> str:
    """Безопасные одинарные кавычки для bash."""
    return "'" + s.replace("'", "'\\''") + "'"


written: list[Path] = []
seen_names: dict[str, int] = {}

for r in results:
    nl = int(r["num_layers"])
    md = int(r["model_dim"])
    mm = int(r["mlp_mult"])
    nh = int(r["num_heads"])
    nkv = int(r["num_kv_heads"])
    ib = int(r["int8_payload_bytes"])
    imb = float(r["int8_payload_mb"])

    base_stem = f"run_golf_L{nl}_d{md}_mlp{mm}_h{nh}_kv{nkv}"
    k = seen_names.get(base_stem, 0) + 1
    seen_names[base_stem] = k
    stem = base_stem if k == 1 else f"{base_stem}_n{k}"

    path = LAUNCH_DIR / f"{stem}.sh"
    run_id = stem.removeprefix("run_")
    experiment_name = (
        f"L={nl} d={md} mlp={mm} nh={nh} nkv={nkv} | "
        f"params={int(r['nparams']):,} int8≈{imb:.3f}MiB"
    )

    if COMET_API_KEY:
        comet_block = f"export COMET_API_KEY={sh_quote(COMET_API_KEY)}\n"
    else:
        comet_block = "# ключ не вшит: см. .comet_api_key в launch_grid_generated/\n"

    body = (
        f"""#!/usr/bin/env bash
set -euo pipefail
# Авто-сгенерировано из check_model_size.ipynb
# int8_payload_bytes={ib}

cd {sh_quote(str(REPO_ROOT))}

"""
        + "\n\n"
        + f"""export RUN_ID={sh_quote(run_id)}
export DATA_PATH={sh_quote(DATA_PATH)}
export TOKENIZER_PATH={sh_quote(TOKENIZER_PATH)}
export VOCAB_SIZE={VOCAB_SIZE}
export COMET_ENABLE={COMET_ENABLE}
{comet_block}export EXPERIMENT_NAME={sh_quote(experiment_name)}
export NUM_LAYERS={nl}
export MODEL_DIM={md}
export NUM_HEADS={nh}
export NUM_KV_HEADS={nkv}
export MLP_MULT={mm}
export USE_COMPILE={USE_COMPILE}

torchrun --standalone --nproc_per_node={NPROC_PER_NODE} train_gpt.py
"""
    )

    path.write_text(body, encoding="utf-8")
    path.chmod(path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
    written.append(path)

print(f"Записано {len(written)} скриптов в {LAUNCH_DIR}:")
for p in written:
    print(" ", p.relative_to(REPO_ROOT))
_keyf = LAUNCH_DIR / ".comet_api_key"
if not COMET_API_KEY and not _keyf.is_file():
    print(
        f"\nComet: создайте {_keyf} (одна строка — API key) или export COMET_API_KEY перед запуском .sh"
    )


Записано 16 скриптов в /home/jovyan/vasiliev/notebooks/parameter-golf/launch_grid_generated:
  launch_grid_generated/run_golf_L8_d448_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L12_d320_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d384_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L16_d256_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L12_d320_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d384_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d320_mlp5_h8_kv4.sh
  launch_grid_generated/run_golf_L12_d256_mlp5_h8_kv4.sh
  launch_grid_generated/run_golf_L16_d256_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d320_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L12_d256_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d320_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L12_d256_mlp3_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d256_mlp5_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d256_mlp4_h8_kv4.sh
  launch_grid_generated/run_golf_L8_d256_mlp3_h8_kv4.sh

Com

In [6]:
import stat
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
LAUNCH_DIR = REPO_ROOT / "launch_grid_generated"
MASTER_NAME = "run_all_golf_grid.sh"

scripts = sorted(LAUNCH_DIR.glob("run_golf_*.sh"))
scripts = [p for p in scripts if p.name != MASTER_NAME]

if not scripts:
    raise RuntimeError(
        f"В {LAUNCH_DIR} нет run_golf_*.sh — сначала выполните ячейку генерации отдельных скриптов"
    )

lines: list[str] = [
    "#!/usr/bin/env bash",
    "set -euo pipefail",
    "# Авто-сгенерировано из check_model_size.ipynb",
    "# Запуск по очереди: при ошибке в одном скрипте — остановка (set -e)",
    'SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"',
    "",
]

for p in scripts:
    name = p.name
    lines.append(f'echo "=== $(date -Iseconds) {name} ==="')
    lines.append(f'bash "$SCRIPT_DIR/{name}"')
    lines.append("")

out_path = LAUNCH_DIR / MASTER_NAME
out_path.write_text("\n".join(lines).rstrip() + "\n", encoding="utf-8")
out_path.chmod(out_path.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print(f"Записан общий раннер ({len(scripts)} шагов):")
print(f"  {out_path.relative_to(REPO_ROOT)}")
print("Запуск из корня репозитория:")
print(f"  bash {out_path.relative_to(REPO_ROOT)}")


Записан общий раннер (16 шагов):
  launch_grid_generated/run_all_golf_grid.sh
Запуск из корня репозитория:
  bash launch_grid_generated/run_all_golf_grid.sh
